# Question 2: 311 Service Requests Analysis

**Project:** Bureau of Tax and Economic Analysis — Quantitative Research

**Data Source:** NYC OpenData 311 Service Requests (API)
**Dataset:** https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9

**Filters applied:**
1. `created_date` between 12/15/2021 and 3/15/2022 (inclusive)
2. `agency_name` = NYPD or HPD
3. `complaint_type` = noise (all subcategories) or illegal parking

---
## Step 3.1: Load Data

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

df = pd.read_csv('../data/311_complaints.csv')
df['created_date'] = pd.to_datetime(df['created_date'])
print(f'Total rows: {len(df):,}')
print(f'Date range: {df["created_date"].min()} to {df["created_date"].max()}')
print(f'Agencies: {sorted(df["agency_name"].unique())}')
print(f'Complaint types: {sorted(df["complaint_type"].unique())}')
print(f'Boroughs: {sorted(df["borough"].unique())}')

---
## Step 3.2: Query URL (Q2a)

The exact SoQL query URL used to pull this data.

In [ ]:
QUERY_URL = (
    'https://data.cityofnewyork.us/resource/erm2-nwe9.json'
    '?$select=unique_key,created_date,agency_name,complaint_type,descriptor,borough,incident_zip,latitude,longitude'
    '&$where=created_date between \'2021-12-15T00:00:00\' and \'2022-03-15T23:59:59\' '
    'AND (agency_name=\'New York City Police Department\' '
    'OR agency_name=\'Department of Housing Preservation and Development\') '
    'AND (starts_with(complaint_type, \'Noise\') '
    'OR complaint_type=\'Illegal Parking\')'
    '&$limit=500000'
)
print(QUERY_URL)
print(f'\nData pulled: {len(df):,} rows | Agencies: {sorted(df["agency_name"].unique())}')

### Why `starts_with(complaint_type, 'Noise')`?

The 311 system categorizes noise into subcategories: "Noise - Residential", "Noise - Commercial", "Noise - Vehicle", "Noise - Street/Sidewalk", etc. An exact match for `complaint_type = 'Noise'` would only capture records handled by the Department of Environmental Protection — missing the majority of noise complaints, which are in NYPD-handled subcategories. Using `starts_with` captures the full scope of noise-related complaints as the question intends.

Additionally, the exact "Noise" type belongs to the **Department of Environmental Protection** (DEP), not NYPD or HPD, so it is correctly excluded by the agency filter.

---
## Step 3.3: Complaint Counts by Agency and Type (Q2b)

In [ ]:
summary = pd.DataFrame({
    'Agency': df.groupby('complaint_type')['agency_name'].first(),
    'Count': df['complaint_type'].value_counts(),
    '% of Total': (df['complaint_type'].value_counts() / len(df) * 100).round(2)
})
summary.loc['TOTAL'] = ['—', len(df), 100.00]
display(summary.style.format({'Count': '{:,.0f}', '% of Total': '{:.2f}%'}))

noise_mask = df['complaint_type'].str.startswith('Noise')
print(f'\nNoise (all subcategories): {noise_mask.sum():,} ({noise_mask.sum()/len(df)*100:.1f}%)')
print(f'Illegal Parking: {(~noise_mask).sum():,} ({(~noise_mask).sum()/len(df)*100:.1f}%)')

---
## Step 3.4: Chart — Complaints by Type (Q2b)

In [ ]:
counts = df['complaint_type'].value_counts().sort_values(ascending=True)

colors = []
for ct in counts.index:
    if ct == 'Illegal Parking':
        colors.append('#1565c0')
    elif 'Residential' in ct:
        colors.append('#d32f2f')
    else:
        colors.append('#ef5350')

fig = go.Figure()
fig.add_trace(go.Bar(
    y=counts.index,
    x=counts.values,
    orientation='h',
    marker_color=colors,
    text=[f'{v:,}' for v in counts.values],
    textposition='outside',
    textfont_size=11,
    cliponaxis=False,
))

x_max = counts.values.max()
fig.update_layout(
    title=f'311 Complaints by Type (Dec 15, 2021 – Mar 15, 2022)<br><sup>Total: {len(df):,} complaints | Agency: NYPD</sup>',
    xaxis_title='Number of Complaints',
    yaxis_title='',
    xaxis_range=[0, x_max * 1.25],
    height=500,
    margin=dict(l=220, r=100, t=80, b=50),
    font=dict(size=11),
    annotations=[
        dict(x=0.85, y=0.05, xref='paper', yref='paper',
             text='Red = Noise subcategories | Blue = Illegal Parking',
             showarrow=False, font=dict(size=10, color='#666'))
    ]
)
fig.show()

---
## Step 3.5: Weekly Trend — Noise vs Illegal Parking (Q2c)

In [ ]:
df['date_bin'] = df['created_date'].dt.to_period('W').astype(str)

weekly = df.groupby(['date_bin', 'complaint_type']).size().unstack(fill_value=0)

noise_cols = [c for c in weekly.columns if c.startswith('Noise')]
weekly['Total Noise'] = weekly[noise_cols].sum(axis=1)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=weekly.index, y=weekly['Total Noise'],
    name='Noise (all subcategories)', mode='lines',
    line=dict(color='#d32f2f', width=2)
))
fig2.add_trace(go.Scatter(
    x=weekly.index, y=weekly['Illegal Parking'],
    name='Illegal Parking', mode='lines',
    line=dict(color='#1565c0', width=2)
))

fig2.update_layout(
    title='Weekly Complaint Trends: Noise vs Illegal Parking<br>'
          '<sup>NYPD, Dec 2021 – Mar 2022</sup>',
    xaxis_title='Week',
    yaxis_title='Complaints per Week',
    height=400,
    font=dict(size=11),
    legend=dict(x=0.01, y=0.99),
)
fig2.show()

---
## Step 3.6: Complaints by Borough

In [ ]:
borough = df[df['borough'] != 'Unspecified'].copy()
borough_counts = borough.groupby('borough').size().reset_index(name='Complaints')
borough_counts = borough_counts.sort_values('Complaints', ascending=False).reset_index(drop=True)
borough_counts['% of Total'] = (borough_counts['Complaints'] / borough_counts['Complaints'].sum() * 100).round(2)
borough_counts['Agency'] = 'NYPD'
display(borough_counts.style.format({'Complaints': '{:,}', '% of Total': '{:.2f}%'}).hide(axis='index'))

---
## Step 3.7: Borough Map

In [ ]:
borough_coords = {
    'BRONX': (40.8448, -73.8648),
    'BROOKLYN': (40.6782, -73.9442),
    'MANHATTAN': (40.7831, -73.9712),
    'QUEENS': (40.7282, -73.7949),
    'STATEN ISLAND': (40.5795, -74.1502)
}

map_df = borough_counts.copy()
map_df['lat'] = map_df['borough'].map(lambda b: borough_coords[b][0])
map_df['lon'] = map_df['borough'].map(lambda b: borough_coords[b][1])

fig_map = px.scatter_map(
    map_df, lat='lat', lon='lon', size='Complaints', color='Complaints',
    hover_name='borough',
    hover_data={'Complaints': True, '% of Total': True, 'lat': False, 'lon': False},
    color_continuous_scale='YlOrRd', size_max=60, zoom=9.5,
    center=dict(lat=40.7128, lon=-73.9560),
    title=f'311 Complaints by Borough (Dec 2021 – Mar 2022)<br>'
          f'<sup>NYPD | Total: {map_df["Complaints"].sum():,} (excludes {len(df) - map_df["Complaints"].sum()} Unspecified)</sup>'
)
fig_map.update_layout(map_style='carto-positron', height=550, font=dict(size=11))
fig_map.update_coloraxes(colorbar_title='Complaints')
fig_map.show()

---
## Step 3.8: Complaint Distribution by Borough — Pie Charts

In [ ]:
borough_names = sorted(borough['borough'].unique())

pie_colors = {
    'Illegal Parking': '#1565c0',
    'Noise - Residential': '#d32f2f',
    'Noise - Street/Sidewalk': '#ef5350',
    'Noise - Vehicle': '#ff7043',
    'Noise - Commercial': '#ff8a65',
    'Noise - Park': '#ffab91',
    'Noise - House of Worship': '#ffccbc'
}

fig_pies = make_subplots(
    rows=1, cols=5,
    specs=[[{'type': 'domain'}] * 5],
    subplot_titles=[b.title() for b in borough_names]
)

for i, b in enumerate(borough_names):
    sub = borough[borough['borough'] == b]
    ct_counts = sub['complaint_type'].value_counts()
    fig_pies.add_trace(go.Pie(
        labels=ct_counts.index,
        values=ct_counts.values,
        name=b.title(),
        marker_colors=[pie_colors.get(ct, '#999') for ct in ct_counts.index],
        textinfo='percent',
        textfont_size=9,
        showlegend=(i == 0),
        hole=0.35,
    ), row=1, col=i + 1)

fig_pies.update_layout(
    title_text='Complaint Type Distribution by Borough (Dec 2021 – Mar 2022)<br>'
                '<sup>NYPD | Blue = Illegal Parking, Red shades = Noise subcategories</sup>',
    height=450,
    font=dict(size=11),
    margin=dict(t=100, b=30),
)
fig_pies.show()

---
## Step 3.9: Borough Breakdown by Complaint Type

In [ ]:
borough_type = borough.groupby(['borough', 'complaint_type']).size().reset_index(name='Count')
borough_type['Agency'] = 'NYPD'
borough_type = borough_type.sort_values(['borough', 'Count'], ascending=[True, False]).reset_index(drop=True)
display(borough_type.style.format({'Count': '{:,}'}).hide(axis='index'))